# GEPA + Llama-3-8B + Adversary (HarmBench)

This notebook runs the `run_with_adversary.py` experiment pipeline.

In [1]:
import os
from getpass import getpass
from run_with_adversary import ExperimentConfig, run_experiment


def ensure_secret(name: str, prompt: str) -> None:
    """Prompt securely when env var is missing."""
    if os.getenv(name):
        return

    value = getpass(prompt).strip()
    if value:
        os.environ[name] = value


ensure_secret("HF_TOKEN", "Enter HF_TOKEN (input hidden): ")
ensure_secret("OPENAI_API_KEY", "Enter OPENAI_API_KEY (input hidden): ")

print('HF_TOKEN set:', bool(os.getenv('HF_TOKEN')))
print('OPENAI_API_KEY set:', bool(os.getenv('OPENAI_API_KEY')))

/root/prompt-optimization/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HF_TOKEN set: False
OPENAI_API_KEY set: False


In [2]:
cfg = ExperimentConfig(
    model_name='meta-llama/Meta-Llama-3-8B-Instruct',
    train_size=100,
    val_size=100,
    max_metric_calls=300,
)
cfg

ExperimentConfig(vllm_base_url='http://127.0.0.1:8000/v1', vllm_api_key='EMPTY', model_name='meta-llama/Meta-Llama-3-8B-Instruct', reflection_model_name='gpt-4o-mini', openai_api_key=None, dataset_name='walledai/HarmBench', dataset_config='standard', dataset_split='train', train_size=100, val_size=100, max_tokens=256, temperature=0.0, max_metric_calls=300, detoxify_model_type='original', detoxify_device='cuda', adversary_online_update_every_calls=8, adversary_warm_start_size=64, output_dir='results', random_seed=42)

In [ ]:
import json
from urllib import error, request


def check_vllm(base_url: str, expected_model: str | None = None, timeout: int = 8) -> None:
    """Fail fast if vLLM OpenAI-compatible endpoint is unreachable."""
    models_url = base_url.rstrip("/") + "/models"
    print(f"Checking vLLM endpoint: {models_url}")

    try:
        with request.urlopen(models_url, timeout=timeout) as resp:
            payload = json.loads(resp.read().decode("utf-8"))
    except error.URLError as exc:
        raise RuntimeError(
            f"Cannot reach vLLM at {models_url}. Start server first and retry. Details: {exc}"
        ) from exc
    except Exception as exc:
        raise RuntimeError(f"vLLM preflight failed: {exc}") from exc

    models = [item.get("id") for item in payload.get("data", []) if isinstance(item, dict)]
    print("vLLM reachable: True")
    print("Served models:", models or "(none)")

    if expected_model:
        if expected_model in models:
            print(f"Expected model found: {expected_model}")
        else:
            print(
                "Warning: expected model not listed. "
                f"expected={expected_model}, available={models}"
            )


check_vllm(cfg.vllm_base_url, cfg.model_name)

In [3]:
summary = run_experiment(cfg)
summary['optimized_metrics']

RuntimeError: OPENAI_API_KEY is required for GEPA reflection model.